In [2]:
# Cell 1: Install libraries
!pip install earthengine-api geemap cartopy geopandas rasterio --quiet
print("✅ Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
✅ Done!


In [3]:
# Cell 2: Authenticate and Initialize GEE
import ee
import geemap

ee.Authenticate(force=True, auth_mode='notebook')
ee.Initialize(project='lulc-467806')
print("✅ GEE Initialized!")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=UBaFNFLvVIRtKIR-Of4mtdBFgeTmvvFVL090vROaGD8&tc=CxlPFDgWn8clr5WrUnrJhJtAVAR8KTh1Q8sj9GTeqQw&cc=K9xEy63vsMnMaydRlAltML6rjK0ZMtj1uarj-HtN_l4

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AeoWuM8BAIuAMeQg808Dx7ekNu7cywVTT1gJT4zokeLTNSTem8ykT2fPtJs

Successfully saved authorization token.
✅ GEE Initialized!


In [4]:
# Cell 3 (fixed): Define AOI with better basemap
aoi = ee.FeatureCollection("FAO/GAUL/2015/level2") \
        .filter(ee.Filter.And(
            ee.Filter.eq('ADM1_NAME', 'Sikkim'),
            ee.Filter.eq('ADM2_NAME', 'East')
        ))

Map = geemap.Map(basemap='HYBRID')  # Use Google basemap instead of OSM
Map.centerObject(aoi, 10)
Map.addLayer(aoi, {'color': 'red'}, 'East Sikkim Boundary')
Map

Map(center=[27.28901296002331, 88.683096910737], controls=(WidgetControl(options=['position', 'transparent_bg'…

In [5]:
# Cell 4 (fixed): Cloud masking that preserves snow pixels

def mask_clouds_preserve_snow(image):
    qa = image.select('QA60')

    # Compute NDSI to identify snow
    ndsi = image.normalizedDifference(['B3', 'B11'])
    is_snow = ndsi.gt(0.4)

    # Cloud bits
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(
                 qa.bitwiseAnd(1 << 11).eq(0))

    # Keep pixel if it is not cloudy OR if it is snow
    final_mask = cloud_mask.Or(is_snow)

    return image.updateMask(final_mask).divide(10000)

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(aoi) \
        .filterDate('2020-01-01', '2024-01-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)) \
        .map(mask_clouds_preserve_snow) \
        .median() \
        .clip(aoi)

print('Sentinel-2 image ready with snow preserved!')

Sentinel-2 image ready with snow preserved!


In [6]:
# Cell 5: Compute spectral indices + terrain + seasonal NDVI

# Main indices
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndsi = s2.normalizedDifference(['B3', 'B11']).rename('NDSI')

# Terrain
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)
slope = ee.Terrain.slope(dem).rename('slope')
aspect = ee.Terrain.aspect(dem).rename('aspect')
elevation = dem.rename('elevation')

# Winter NDVI
s2_winter = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(aoi) \
        .filterDate('2021-12-01', '2022-02-28') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)) \
        .map(mask_clouds_preserve_snow) \
        .median() \
        .clip(aoi)

ndvi_winter = s2_winter.normalizedDifference(['B8', 'B4']).rename('NDVI_winter')

# Monsoon NDVI  <-- ADD HERE
s2_monsoon = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(aoi) \
        .filterDate('2021-06-01', '2021-09-30') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)) \
        .map(mask_clouds_preserve_snow) \
        .median() \
        .clip(aoi)

ndvi_monsoon = s2_monsoon.normalizedDifference(['B8', 'B4']).rename('NDVI_monsoon')
ndvi_diff = ndvi_monsoon.subtract(ndvi_winter).rename('NDVI_diff')

# Final stack
image = s2.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']) \
          .addBands([ndvi, ndwi, ndsi, elevation, slope, aspect,
                     ndvi_winter, ndvi_monsoon, ndvi_diff])

print('Bands in final image:', image.bandNames().getInfo())

Bands in final image: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDSI', 'elevation', 'slope', 'aspect', 'NDVI_winter', 'NDVI_monsoon', 'NDVI_diff']


In [7]:
# Cell 6: Load ESA WorldCover 2021 and remap to your classes
worldcover = ee.ImageCollection("ESA/WorldCover/v200") \
               .first() \
               .clip(aoi)

remapped = worldcover.remap(
    [10, 20, 30, 40, 50, 60, 80, 90],
    [1,  1,  2,  2,  3,  4,  0,  0]
).rename('landcover')

# Sample stratified points from WorldCover
extra_points = remapped.stratifiedSample(
    numPoints=200,
    classBand='landcover',
    region=aoi,
    scale=10,
    seed=42,
    geometries=True
)

print('Extra points from WorldCover:', extra_points.size().getInfo())
print('Properties:', extra_points.first().propertyNames().getInfo())

Extra points from WorldCover: 1000
Properties: ['landcover', 'system:index']


In [8]:
# Cell 7: Merge original training points with WorldCover points
original_training = ee.FeatureCollection("projects/lulc-467806/assets/training_gangtok")

# Check property name of original training points
print('Original points:', original_training.size().getInfo())
print('Original properties:', original_training.first().propertyNames().getInfo())

Original points: 346
Original properties: ['Class', 'system:index']


In [9]:
# Cell 8: Harmonize property names and merge
# Rename 'Class' to 'landcover' in original training points
original_training = original_training.map(
    lambda f: f.set('landcover', f.get('Class'))
)

# Merge both collections
merged_training = original_training.merge(extra_points)

print('Original points:', original_training.size().getInfo())
print('WorldCover points:', extra_points.size().getInfo())
print('Total merged points:', merged_training.size().getInfo())
print('Class distribution:', merged_training.aggregate_histogram('landcover').getInfo())

Original points: 346
WorldCover points: 1000
Total merged points: 1346
Class distribution: {'0': 250, '1': 277, '2': 250, '3': 299, '4': 261, 'null': 9}


In [10]:
# Cell 9: Remove null values and verify
merged_training = merged_training.filter(ee.Filter.notNull(['landcover']))

print('Final training points after cleaning:', merged_training.size().getInfo())
print('Class distribution:', merged_training.aggregate_histogram('landcover').getInfo())


Final training points after cleaning: 1337
Class distribution: {'0': 250, '1': 277, '2': 250, '3': 299, '4': 261}


In [11]:
# Cell 10: Sample image values at training points
feature_names = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12',
                 'NDVI', 'NDWI', 'NDSI',
                 'elevation', 'slope', 'aspect',
                 'NDVI_winter', 'NDVI_monsoon', 'NDVI_diff']

training_samples = image.sampleRegions(
    collection=merged_training,
    properties=['landcover'],
    scale=10,
    tileScale=2
)

print('Training samples:', training_samples.size().getInfo())

Training samples: 1308


In [13]:
# Cell 11: Split into train/validation sets
training_samples = training_samples.randomColumn('random', 42)

train = training_samples.filter(ee.Filter.lt('random', 0.7))
validation = training_samples.filter(ee.Filter.gte('random', 0.7))

print('Training set:', train.size().getInfo())
print('Validation set:', validation.size().getInfo())

Training set: 883
Validation set: 425


In [14]:
# Cell 12: Train classifiers
feature_names = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12',
                 'NDVI', 'NDWI', 'NDSI',
                 'elevation', 'slope', 'aspect',
                 'NDVI_winter', 'NDVI_monsoon', 'NDVI_diff']

# Random Forest
rf = ee.Classifier.smileRandomForest(
    numberOfTrees=200,
    variablesPerSplit=3,
    minLeafPopulation=2,
    seed=42
).train(
    features=train,
    classProperty='landcover',
    inputProperties=feature_names
)



# Gradient Tree Boost
gtb = ee.Classifier.smileGradientTreeBoost(
    numberOfTrees=200,
    shrinkage=0.05,
    seed=42
).train(
    features=train,
    classProperty='landcover',
    inputProperties=feature_names
)

print('Both classifiers trained!')

Both classifiers trained!


In [16]:
# Cell 13: Evaluate classifiers on validation set

# Classify validation set with each classifier
val_rf  = validation.classify(rf)
val_gtb = validation.classify(gtb)

# Confusion matrices
cm_rf  = val_rf.errorMatrix('landcover', 'classification')
cm_gtb = val_gtb.errorMatrix('landcover', 'classification')

# Overall accuracies
oa_rf  = cm_rf.accuracy().getInfo()
oa_gtb = cm_gtb.accuracy().getInfo()

# Kappa coefficients
kappa_rf  = cm_rf.kappa().getInfo()
kappa_gtb = cm_gtb.kappa().getInfo()

print('--- Random Forest ---')
print('Overall Accuracy:', round(oa_rf, 4))
print('Kappa:', round(kappa_rf, 4))


print('\n--- Gradient Tree Boost ---')
print('Overall Accuracy:', round(oa_gtb, 4))
print('Kappa:', round(kappa_gtb, 4))

--- Random Forest ---
Overall Accuracy: 0.7576
Kappa: 0.6956

--- Gradient Tree Boost ---
Overall Accuracy: 0.76
Kappa: 0.699


In [17]:
# Per-class accuracy breakdown
print('--- GTB Confusion Matrix ---')
print(cm_gtb.getInfo())

print('\nProducers Accuracy (recall per class):')
print(cm_gtb.producersAccuracy().getInfo())

print('\nConsumers Accuracy (precision per class):')
print(cm_gtb.consumersAccuracy().getInfo())

--- GTB Confusion Matrix ---
[[72, 10, 2, 3, 8], [2, 72, 4, 0, 1], [1, 21, 47, 6, 6], [0, 5, 4, 88, 2], [9, 9, 4, 5, 44]]

Producers Accuracy (recall per class):
[[0.7578947368421053], [0.9113924050632911], [0.5802469135802469], [0.8888888888888888], [0.6197183098591549]]

Consumers Accuracy (precision per class):
[[0.8571428571428571, 0.6153846153846154, 0.7704918032786885, 0.8627450980392157, 0.7213114754098361]]


In [18]:
# Cell 14: Classify full image and visualize

classified = image.classify(rf)

palette = ['#1a6fba',  # 0 - Water/Snow
           '#228b22',  # 1 - Vegetation
           '#90ee90',  # 2 - Grassland/Cropland
           '#d2691e',  # 3 - Built-up
           '#f5deb3']  # 4 - Bare land

vis_params = {
    'min': 0,
    'max': 4,
    'palette': palette
}

Map.addLayer(classified, vis_params, 'LULC Classification (RF)')
Map

Map(bottom=110704.0, center=[27.28901296002331, 88.683096910737], controls=(WidgetControl(options=['position',…

In [19]:
# Export classified raster to Google Drive
task = ee.batch.Export.image.toDrive(
    image=classified,
    description='LULC_EastSikkim_2024',
    folder='GEE_Exports',
    fileNamePrefix='LULC_EastSikkim_RF',
    region=aoi.geometry(),
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task.start()
print("Export started! Check Tasks tab in GEE Code Editor.")
print("Status:", task.status())

Export started! Check Tasks tab in GEE Code Editor.
Status: {'state': 'READY', 'description': 'LULC_EastSikkim_2024', 'priority': 100, 'creation_timestamp_ms': 1777339172438, 'update_timestamp_ms': 1777339172438, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'TEFVIHAOUKRGRUXH5UOUM5MR', 'name': 'projects/lulc-467806/operations/TEFVIHAOUKRGRUXH5UOUM5MR'}


In [20]:
# ============================================================
# COMPREHENSIVE MODEL EVALUATION — ALL PLOTS
# Run after Cell 13
# Output: /content/eval_plots/
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
import os

os.makedirs('/content/eval_plots', exist_ok=True)
OUT = '/content/eval_plots'

# ─────────────────────────────────────────────
# GLOBAL STYLE
# ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#333333',
    'axes.linewidth':    1.2,
    'axes.grid':         False,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'axes.labelweight':  'bold',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'savefig.facecolor': 'white',
    'savefig.dpi':       300,
})

CLASS_NAMES  = ['Water/Snow', 'Vegetation', 'Grassland/Crop', 'Built-up', 'Bare Land']
CLASS_COLORS = ['#1a6fba', '#228b22', '#90ee90', '#8B0000', '#d2a679']
RF_COLOR     = '#1b4f72'
GTB_COLOR    = '#7b241c'
SHORT        = ['Water/\nSnow', 'Veget-\nation', 'Grass/\nCrop', 'Built-\nup', 'Bare\nLand']
N            = 5

# ─────────────────────────────────────────────
# FETCH FROM PRIOR CELLS — NO FALLBACKS
# ─────────────────────────────────────────────
print("Fetching metrics from GEE...")

cm_rf_array  = np.array(cm_rf.getInfo())
cm_gtb_array = np.array(cm_gtb.getInfo())

oa_rf_val     = round(oa_rf  * 100, 2)
oa_gtb_val    = round(oa_gtb * 100, 2)
kappa_rf_val  = round(kappa_rf,  4)
kappa_gtb_val = round(kappa_gtb, 4)

pa_rf_vals  = [v[0] * 100 for v in cm_rf.producersAccuracy().getInfo()]
ca_rf_vals  = [v * 100    for v in cm_rf.consumersAccuracy().getInfo()[0]]
pa_gtb_vals = [v[0] * 100 for v in cm_gtb.producersAccuracy().getInfo()]
ca_gtb_vals = [v * 100    for v in cm_gtb.consumersAccuracy().getInfo()[0]]

hist         = merged_training.aggregate_histogram('landcover').getInfo()
class_counts = [int(hist.get(str(i), 0)) for i in range(N)]

print("✅ All GEE metrics fetched.\n")

# ─────────────────────────────────────────────
# DERIVED METRICS
# ─────────────────────────────────────────────
def f1_scores(pa, ca):
    pa, ca = np.array(pa) / 100, np.array(ca) / 100
    return list(2 * pa * ca / np.where((pa + ca) == 0, 1, pa + ca) * 100)

f1_rf  = f1_scores(pa_rf_vals,  ca_rf_vals)
f1_gtb = f1_scores(pa_gtb_vals, ca_gtb_vals)

omis_rf  = [100 - v for v in pa_rf_vals]
comm_rf  = [100 - v for v in ca_rf_vals]
omis_gtb = [100 - v for v in pa_gtb_vals]
comm_gtb = [100 - v for v in ca_gtb_vals]

# ─────────────────────────────────────────────
# SAVE HELPER
# ─────────────────────────────────────────────
def save(filename):
    plt.savefig(f'{OUT}/{filename}', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  ✅  {filename}")


# ════════════════════════════════════════════════════════════
# CM_01 — Confusion Matrix (Raw Counts) — Random Forest
# ════════════════════════════════════════════════════════════
def plot_cm_counts(cm, color, filename):
    cmap = LinearSegmentedColormap.from_list('c', ['#ffffff', color], N=256)
    fig, ax = plt.subplots(figsize=(8, 7), facecolor='white')
    im   = ax.imshow(cm, cmap=cmap, aspect='auto')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Sample Count', fontsize=10, fontweight='bold')
    cbar.ax.tick_params(labelsize=9)
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_yticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Predicted Class', fontsize=11, fontweight='bold', labelpad=8)
    ax.set_ylabel('Actual Class',    fontsize=11, fontweight='bold', labelpad=8)
    ax.tick_params(length=0)
    vmax = cm.max()
    for i in range(N):
        for j in range(N):
            fc = 'white' if cm[i, j] > vmax * 0.55 else '#111111'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=12, color=fc, fontweight='bold')
    for i in range(N):
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                     fill=False, edgecolor='gold', linewidth=2.5))
    ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    plt.tight_layout()
    save(filename)

plot_cm_counts(cm_rf_array,  RF_COLOR,  'CM_01_RF_ConfusionMatrix_Counts.png')
plot_cm_counts(cm_gtb_array, GTB_COLOR, 'CM_02_GTB_ConfusionMatrix_Counts.png')


# ════════════════════════════════════════════════════════════
# CM_03 — Confusion Matrix (Row-Normalized %) — RF & GTB
# ════════════════════════════════════════════════════════════
def plot_cm_norm(cm, color, filename):
    norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    cmap = LinearSegmentedColormap.from_list('c', ['#ffffff', color], N=256)
    fig, ax = plt.subplots(figsize=(8, 7), facecolor='white')
    im   = ax.imshow(norm, cmap=cmap, aspect='auto', vmin=0, vmax=100)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Row %', fontsize=10, fontweight='bold')
    cbar.ax.tick_params(labelsize=9)
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_yticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Predicted Class', fontsize=11, fontweight='bold', labelpad=8)
    ax.set_ylabel('Actual Class',    fontsize=11, fontweight='bold', labelpad=8)
    ax.tick_params(length=0)
    for i in range(N):
        for j in range(N):
            fc = 'white' if norm[i, j] > 55 else '#111111'
            ax.text(j, i, f'{norm[i, j]:.1f}%', ha='center', va='center',
                    fontsize=10, color=fc, fontweight='bold')
    for i in range(N):
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                     fill=False, edgecolor='gold', linewidth=2.5))
    ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    plt.tight_layout()
    save(filename)

plot_cm_norm(cm_rf_array,  RF_COLOR,  'CM_03_RF_ConfusionMatrix_Normalized.png')
plot_cm_norm(cm_gtb_array, GTB_COLOR, 'CM_04_GTB_ConfusionMatrix_Normalized.png')


# ════════════════════════════════════════════════════════════
# ACC_05 — Overall Accuracy Comparison (RF vs GTB)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
models = ['Random Forest', 'Gradient\nTree Boost']
vals   = [oa_rf_val, oa_gtb_val]
bars   = ax.bar(models, vals, color=[RF_COLOR, GTB_COLOR],
                width=0.40, edgecolor='white', linewidth=0)
ax.set_ylim(0, 110)
ax.set_xlabel('Classifier',           fontsize=11, fontweight='bold')
ax.set_ylabel('Overall Accuracy (%)', fontsize=11, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{v:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
save('ACC_05_OverallAccuracy_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# ACC_06 — Kappa Coefficient Comparison (RF vs GTB)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
kvals = [kappa_rf_val, kappa_gtb_val]
bars  = ax.bar(models, kvals, color=[RF_COLOR, GTB_COLOR],
               width=0.40, edgecolor='white', linewidth=0)
ax.set_ylim(0, 1.05)
ax.set_xlabel('Classifier',                fontsize=11, fontweight='bold')
ax.set_ylabel("Cohen's Kappa Coefficient", fontsize=11, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
for bar, v in zip(bars, kvals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{v:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
save('ACC_06_Kappa_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# PA_07 / PA_08 — Producer's & Consumer's Accuracy per Class
# ════════════════════════════════════════════════════════════
def plot_pa_ca(pa, ca, filename):
    x = np.arange(N); w = 0.30
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
    b1 = ax.bar(x - w/2, pa, w, label="Producer's Accuracy (Recall)",
                color='#1a6fba', edgecolor='white', linewidth=0)
    b2 = ax.bar(x + w/2, ca, w, label="Consumer's Accuracy (Precision)",
                color='#e67e22', edgecolor='white', linewidth=0)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Land Cover Class', fontsize=11, fontweight='bold')
    ax.set_ylabel('Accuracy (%)',     fontsize=11, fontweight='bold')
    ax.set_ylim(0, 115)
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=0)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                f'{h:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.tight_layout()
    save(filename)

plot_pa_ca(pa_rf_vals,  ca_rf_vals,  'PA_07_RF_ProducerConsumerAccuracy_PerClass.png')
plot_pa_ca(pa_gtb_vals, ca_gtb_vals, 'PA_08_GTB_ProducerConsumerAccuracy_PerClass.png')


# ════════════════════════════════════════════════════════════
# PA_09 — Producer's Accuracy Comparison (RF vs GTB)
# ════════════════════════════════════════════════════════════
x = np.arange(N); w = 0.30
fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
b1 = ax.bar(x - w/2, pa_rf_vals,  w, label='Random Forest',
            color=RF_COLOR,  edgecolor='white', linewidth=0)
b2 = ax.bar(x + w/2, pa_gtb_vals, w, label='Gradient Tree Boost',
            color=GTB_COLOR, edgecolor='white', linewidth=0)
ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
ax.set_xlabel('Land Cover Class',        fontsize=11, fontweight='bold')
ax.set_ylabel("Producer's Accuracy (%)", fontsize=11, fontweight='bold')
ax.set_ylim(0, 115); ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
            f'{h:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
save('PA_09_ProducersAccuracy_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# PA_10 — Consumer's Accuracy Comparison (RF vs GTB)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
b1 = ax.bar(x - w/2, ca_rf_vals,  w, label='Random Forest',
            color=RF_COLOR,  edgecolor='white', linewidth=0)
b2 = ax.bar(x + w/2, ca_gtb_vals, w, label='Gradient Tree Boost',
            color=GTB_COLOR, edgecolor='white', linewidth=0)
ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
ax.set_xlabel('Land Cover Class',         fontsize=11, fontweight='bold')
ax.set_ylabel("Consumer's Accuracy (%)",  fontsize=11, fontweight='bold')
ax.set_ylim(0, 115); ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
            f'{h:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
save('PA_10_ConsumersAccuracy_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# F1_11 / F1_12 — F1-Score per Class (RF, GTB)
# ════════════════════════════════════════════════════════════
def plot_f1(f1s, color, filename):
    x = np.arange(N)
    fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
    bars = ax.bar(x, f1s, color=color, width=0.45, edgecolor='white', linewidth=0)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Land Cover Class', fontsize=11, fontweight='bold')
    ax.set_ylabel('F1-Score (%)',     fontsize=11, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 1,
                f'{v:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    plt.tight_layout()
    save(filename)

plot_f1(f1_rf,  RF_COLOR,  'F1_11_RF_F1Score_PerClass.png')
plot_f1(f1_gtb, GTB_COLOR, 'F1_12_GTB_F1Score_PerClass.png')


# ════════════════════════════════════════════════════════════
# F1_13 — F1-Score Comparison (RF vs GTB)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')
b1 = ax.bar(x - w/2, f1_rf,  w, label='Random Forest',
            color=RF_COLOR,  edgecolor='white', linewidth=0)
b2 = ax.bar(x + w/2, f1_gtb, w, label='Gradient Tree Boost',
            color=GTB_COLOR, edgecolor='white', linewidth=0)
ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
ax.set_xlabel('Land Cover Class', fontsize=11, fontweight='bold')
ax.set_ylabel('F1-Score (%)',     fontsize=11, fontweight='bold')
ax.set_ylim(0, 110); ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
            f'{h:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
plt.tight_layout()
save('F1_13_F1Score_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# ERR_14 / ERR_15 — Omission & Commission Error (RF vs GTB)
# ════════════════════════════════════════════════════════════
def plot_errors(rf_vals, gtb_vals, ylabel, filename):
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
    b1 = ax.bar(x - w/2, rf_vals,  w, label='Random Forest',
                color=RF_COLOR,  edgecolor='white', linewidth=0)
    b2 = ax.bar(x + w/2, gtb_vals, w, label='Gradient Tree Boost',
                color=GTB_COLOR, edgecolor='white', linewidth=0)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Land Cover Class', fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel,             fontsize=11, fontweight='bold')
    ax.set_ylim(0, 55); ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    plt.tight_layout()
    save(filename)

plot_errors(omis_rf, omis_gtb, 'Omission Error (%)',   'ERR_14_OmissionError_RF_vs_GTB.png')
plot_errors(comm_rf, comm_gtb, 'Commission Error (%)', 'ERR_15_CommissionError_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# RAD_16 / RAD_17 — Radar Charts (RF, GTB)
# ════════════════════════════════════════════════════════════
def plot_radar(pa, ca, f1s, color, filename):
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]
    def rv(v): vv = list(v); vv.append(vv[0]); return vv
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), facecolor='white')
    ax.set_facecolor('white')
    for ring in [60, 70, 80, 90, 100]:
        ax.plot(angles, [ring] * (N + 1), '--', color='#cccccc', linewidth=0.7)
        ax.text(0, ring, f'{ring}%', fontsize=7, color='#888888', ha='center', va='bottom')
    for vals, lbl, lw in [
            (pa,  "Producer's Accuracy", 2.2),
            (ca,  "Consumer's Accuracy", 2.2),
            (f1s, "F1-Score",            2.5)]:
        ax.plot(angles, rv(vals), color=color, linewidth=lw, label=lbl)
        ax.fill(angles, rv(vals), color=color, alpha=0.07)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(CLASS_NAMES, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 100); ax.set_yticks([])
    ax.legend(loc='lower right', bbox_to_anchor=(1.40, -0.08), fontsize=10, framealpha=0.9)
    plt.tight_layout()
    save(filename)

plot_radar(pa_rf_vals,  ca_rf_vals,  f1_rf,  RF_COLOR,  'RAD_16_RF_RadarChart_AccuracyMetrics.png')
plot_radar(pa_gtb_vals, ca_gtb_vals, f1_gtb, GTB_COLOR, 'RAD_17_GTB_RadarChart_AccuracyMetrics.png')


# ════════════════════════════════════════════════════════════
# HM_18 / HM_19 — Metrics Heatmap per Class (RF, GTB)
# ════════════════════════════════════════════════════════════
metric_rows  = ["Producer's Accuracy", "Consumer's Accuracy", "F1-Score",
                "Omission Error", "Commission Error"]

for data_rows, color, fname in [
    ([pa_rf_vals,  ca_rf_vals,  f1_rf,  omis_rf,  comm_rf],  RF_COLOR,  'HM_18_RF_MetricsHeatmap_PerClass.png'),
    ([pa_gtb_vals, ca_gtb_vals, f1_gtb, omis_gtb, comm_gtb], GTB_COLOR, 'HM_19_GTB_MetricsHeatmap_PerClass.png')]:

    data = np.array(data_rows)
    cmap = LinearSegmentedColormap.from_list('c', ['#ffffff', color], N=256)
    fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')
    im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=0, vmax=100)
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.03)
    cbar.set_label('Value (%)', fontsize=10, fontweight='bold')
    cbar.ax.tick_params(labelsize=9)
    ax.set_xticks(range(N)); ax.set_yticks(range(len(metric_rows)))
    ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_yticklabels(metric_rows, fontsize=10)
    ax.set_xlabel('Land Cover Class', fontsize=11, fontweight='bold')
    ax.set_ylabel('Metric',           fontsize=11, fontweight='bold')
    ax.tick_params(length=0)
    ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    for i in range(len(metric_rows)):
        for j in range(N):
            fc = 'white' if data[i, j] > 55 else '#111111'
            ax.text(j, i, f'{data[i, j]:.1f}', ha='center', va='center',
                    fontsize=11, color=fc, fontweight='bold')
    plt.tight_layout()
    save(fname)


# ════════════════════════════════════════════════════════════
# PRED_20 — Predicted vs True Class Distribution (RF)
# PRED_21 — Predicted vs True Class Distribution (GTB)
# ════════════════════════════════════════════════════════════
def get_pred_distribution(val_classified, true_col='landcover', pred_col='classification'):
    """Extract predicted and true class counts from a classified FeatureCollection."""
    true_hist = val_classified.aggregate_histogram(true_col).getInfo()
    pred_hist = val_classified.aggregate_histogram(pred_col).getInfo()
    true_counts = [int(true_hist.get(str(i), 0)) for i in range(N)]
    pred_counts = [int(pred_hist.get(str(i), 0)) for i in range(N)]
    return true_counts, pred_counts

def plot_pred_vs_true(true_counts, pred_counts, model_color, filename):
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
    xi = np.arange(N); w = 0.30
    b1 = ax.bar(xi - w/2, true_counts, w, label='True Class',
                color='#555555', edgecolor='white', linewidth=0)
    b2 = ax.bar(xi + w/2, pred_counts, w, label='Predicted Class',
                color=model_color, edgecolor='white', linewidth=0)
    ax.set_xticks(xi); ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Land Cover Class',    fontsize=11, fontweight='bold')
    ax.set_ylabel('Sample Count',        fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                str(int(h)), ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.tight_layout()
    save(filename)

true_rf,  pred_rf  = get_pred_distribution(val_rf)
true_gtb, pred_gtb = get_pred_distribution(val_gtb)

plot_pred_vs_true(true_rf,  pred_rf,  RF_COLOR,  'PRED_20_RF_TrueVsPredicted_ClassCounts.png')
plot_pred_vs_true(true_gtb, pred_gtb, GTB_COLOR, 'PRED_21_GTB_TrueVsPredicted_ClassCounts.png')


# ════════════════════════════════════════════════════════════
# PRED_22 — Prediction Agreement (% correctly predicted per class, RF vs GTB)
# ════════════════════════════════════════════════════════════
# Diagonal of normalized confusion matrix = per-class recall
recall_rf  = [cm_rf_array[i, i]  / cm_rf_array[i, :].sum()  * 100 for i in range(N)]
recall_gtb = [cm_gtb_array[i, i] / cm_gtb_array[i, :].sum() * 100 for i in range(N)]

fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
b1 = ax.bar(x - w/2, recall_rf,  w, label='Random Forest',
            color=RF_COLOR,  edgecolor='white', linewidth=0)
b2 = ax.bar(x + w/2, recall_gtb, w, label='Gradient Tree Boost',
            color=GTB_COLOR, edgecolor='white', linewidth=0)
ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
ax.set_xlabel('Land Cover Class',              fontsize=11, fontweight='bold')
ax.set_ylabel('Correctly Predicted Samples (%)', fontsize=11, fontweight='bold')
ax.set_ylim(0, 115); ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
            f'{h:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
save('PRED_22_PerClassPredictionAccuracy_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# DIST_23 — Training Sample Distribution (Pie)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 8), facecolor='white')
wedges, texts, autotexts = ax.pie(
    class_counts, labels=CLASS_NAMES, colors=CLASS_COLORS,
    autopct='%1.1f%%', startangle=140, pctdistance=0.78,
    wedgeprops=dict(edgecolor='white', linewidth=2.5),
    textprops=dict(fontsize=11))
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold'); at.set_color('white')
ax.add_patch(plt.Circle((0, 0), 0.45, color='white'))
ax.text(0, 0, f'Total\n{sum(class_counts):,}', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#333333')
ax.legend(wedges, [f'{l}  ({c:,})' for l, c in zip(CLASS_NAMES, class_counts)],
          loc='lower center', bbox_to_anchor=(0.5, -0.10), ncol=3, fontsize=10)
plt.tight_layout()
save('DIST_23_TrainingSampleDistribution_Pie.png')


# ════════════════════════════════════════════════════════════
# DIST_24 — Training Sample Proportion (Bar)
# ════════════════════════════════════════════════════════════
total = sum(class_counts)
pct   = [c / total * 100 for c in class_counts]
fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
bars = ax.bar(CLASS_NAMES, pct, color=CLASS_COLORS, edgecolor='white', linewidth=0, width=0.55)
ax.set_xlabel('Land Cover Class',      fontsize=11, fontweight='bold')
ax.set_ylabel('Sample Proportion (%)', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(pct) * 1.25)
ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
for bar, p in zip(bars, pct):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f'{p:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
save('DIST_24_TrainingSampleProportion_Bar.png')


# ════════════════════════════════════════════════════════════
# TBL_25 — Summary Accuracy Table (RF vs GTB)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 3.5), facecolor='white')
ax.axis('off')
col_labels = ['Classifier', 'OA (%)', 'Kappa',
              'W/S F1', 'Veg F1', 'G/C F1', 'Blt F1', 'Bare F1']
row_data = [
    ['Random Forest',
     f'{oa_rf_val:.2f}', f'{kappa_rf_val:.4f}'] + [f'{v:.1f}' for v in f1_rf],
    ['Gradient Tree Boost',
     f'{oa_gtb_val:.2f}', f'{kappa_gtb_val:.4f}'] + [f'{v:.1f}' for v in f1_gtb],
]
tbl = ax.table(cellText=row_data, colLabels=col_labels, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.0, 3.2)
for c in range(len(col_labels)):
    tbl[(0, c)].set_facecolor('#1b2a4a')
    tbl[(0, c)].set_text_props(color='white', fontweight='bold')
for r in range(1, 3):
    for c in range(len(col_labels)):
        tbl[(r, c)].set_facecolor('#eaf0fb' if r % 2 == 1 else '#ffffff')
best_row = 1 if oa_rf_val >= oa_gtb_val else 2
tbl[(best_row, 1)].set_facecolor('#c8f7c5')
tbl[(best_row, 1)].set_text_props(fontweight='bold', color='#155724')
plt.tight_layout()
save('TBL_25_AccuracySummary_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# TBL_26 — Per-Class Metrics Table (RF vs GTB)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(15, 5), facecolor='white')
ax.axis('off')
col_labels2 = ['Class', 'RF PA (%)', 'RF CA (%)', 'RF F1 (%)',
               'GTB PA (%)', 'GTB CA (%)', 'GTB F1 (%)']
row_data2 = [
    [CLASS_NAMES[i],
     f'{pa_rf_vals[i]:.1f}',  f'{ca_rf_vals[i]:.1f}',  f'{f1_rf[i]:.1f}',
     f'{pa_gtb_vals[i]:.1f}', f'{ca_gtb_vals[i]:.1f}', f'{f1_gtb[i]:.1f}']
    for i in range(N)
]
tbl2 = ax.table(cellText=row_data2, colLabels=col_labels2, cellLoc='center', loc='center')
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.0, 3.0)
for c in range(len(col_labels2)):
    tbl2[(0, c)].set_facecolor('#1b2a4a')
    tbl2[(0, c)].set_text_props(color='white', fontweight='bold')
for r in range(1, N + 1):
    tbl2[(r, 0)].set_facecolor(CLASS_COLORS[r - 1])
    tbl2[(r, 0)].set_text_props(color='white', fontweight='bold')
    for c in range(1, len(col_labels2)):
        tbl2[(r, c)].set_facecolor('#f7f9fc' if r % 2 == 1 else '#ffffff')
plt.tight_layout()
save('TBL_26_PerClassMetrics_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# TBL_27 — Error Metrics Table (Omission & Commission)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5), facecolor='white')
ax.axis('off')
col_labels3 = ['Class', 'RF Omission (%)', 'RF Commission (%)',
               'GTB Omission (%)', 'GTB Commission (%)']
row_data3 = [
    [CLASS_NAMES[i],
     f'{omis_rf[i]:.1f}', f'{comm_rf[i]:.1f}',
     f'{omis_gtb[i]:.1f}', f'{comm_gtb[i]:.1f}']
    for i in range(N)
]
tbl3 = ax.table(cellText=row_data3, colLabels=col_labels3, cellLoc='center', loc='center')
tbl3.auto_set_font_size(False); tbl3.set_fontsize(11); tbl3.scale(1.0, 3.0)
for c in range(len(col_labels3)):
    tbl3[(0, c)].set_facecolor('#1b2a4a')
    tbl3[(0, c)].set_text_props(color='white', fontweight='bold')
for r in range(1, N + 1):
    tbl3[(r, 0)].set_facecolor(CLASS_COLORS[r - 1])
    tbl3[(r, 0)].set_text_props(color='white', fontweight='bold')
    for c in range(1, len(col_labels3)):
        tbl3[(r, c)].set_facecolor('#f7f9fc' if r % 2 == 1 else '#ffffff')
plt.tight_layout()
save('TBL_27_ErrorMetrics_RF_vs_GTB.png')


# ════════════════════════════════════════════════════════════
# ROC & PRECISION-RECALL CURVES
# Uses MULTIPROBABILITY output mode from GEE classifiers
# ════════════════════════════════════════════════════════════
print("\nFetching probability outputs for ROC / PR curves...")

try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize

    # Reclassify validation set with probability output
    rf_prob  = rf.setOutputMode('MULTIPROBABILITY')
    gtb_prob = gtb.setOutputMode('MULTIPROBABILITY')

    val_rf_prob  = validation.classify(rf_prob)
    val_gtb_prob = validation.classify(gtb_prob)

    def fetch_probs(classified_fc, n_classes=5):
        feats = classified_fc.getInfo()['features']
        y_true, y_prob = [], []
        for f in feats:
            props = f['properties']
            y_true.append(int(props['landcover']))
            y_prob.append(props['classification'])   # list of n_classes probabilities
        return np.array(y_true), np.array(y_prob)

    y_true_rf,  probs_rf  = fetch_probs(val_rf_prob)
    y_true_gtb, probs_gtb = fetch_probs(val_gtb_prob)

    y_bin_rf  = label_binarize(y_true_rf,  classes=list(range(N)))
    y_bin_gtb = label_binarize(y_true_gtb, classes=list(range(N)))

    print("  Probability data fetched. Plotting ROC and PR curves...")

    # ── ROC_28 — RF ROC Curves per Class ─────────────────────
    fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
    for i in range(N):
        fpr, tpr, _ = roc_curve(y_bin_rf[:, i], probs_rf[:, i])
        ax.plot(fpr, tpr, color=CLASS_COLORS[i], linewidth=2,
                label=f'{CLASS_NAMES[i]}  AUC={auc(fpr, tpr):.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate',  fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.legend(fontsize=9, loc='lower right')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    plt.tight_layout()
    save('ROC_28_RF_ROCCurve_PerClass.png')

    # ── ROC_29 — GTB ROC Curves per Class ────────────────────
    fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
    for i in range(N):
        fpr, tpr, _ = roc_curve(y_bin_gtb[:, i], probs_gtb[:, i])
        ax.plot(fpr, tpr, color=CLASS_COLORS[i], linewidth=2,
                label=f'{CLASS_NAMES[i]}  AUC={auc(fpr, tpr):.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate',  fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.legend(fontsize=9, loc='lower right')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    plt.tight_layout()
    save('ROC_29_GTB_ROCCurve_PerClass.png')

    # ── ROC_30 — Macro-Average ROC (RF vs GTB) ───────────────
    fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
    for probs, y_bin, lbl, color in [
            (probs_rf,  y_bin_rf,  'Random Forest',       RF_COLOR),
            (probs_gtb, y_bin_gtb, 'Gradient Tree Boost', GTB_COLOR)]:
        all_fpr = np.unique(np.concatenate(
            [roc_curve(y_bin[:, i], probs[:, i])[0] for i in range(N)]))
        mean_tpr = np.mean([
            np.interp(all_fpr, *roc_curve(y_bin[:, i], probs[:, i])[:2])
            for i in range(N)], axis=0)
        macro_auc = auc(all_fpr, mean_tpr)
        ax.plot(all_fpr, mean_tpr, color=color, linewidth=2.5,
                label=f'{lbl}  Macro AUC={macro_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate',  fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.legend(fontsize=10, loc='lower right')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    plt.tight_layout()
    save('ROC_30_MacroAverageROC_RF_vs_GTB.png')

    # ── ROC_31 — AUC per Class Bar (RF vs GTB) ───────────────
    auc_rf  = [auc(*roc_curve(y_bin_rf[:,  i], probs_rf[:,  i])[:2]) for i in range(N)]
    auc_gtb = [auc(*roc_curve(y_bin_gtb[:, i], probs_gtb[:, i])[:2]) for i in range(N)]
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
    b1 = ax.bar(x - w/2, auc_rf,  w, label='Random Forest',
                color=RF_COLOR,  edgecolor='white', linewidth=0)
    b2 = ax.bar(x + w/2, auc_gtb, w, label='Gradient Tree Boost',
                color=GTB_COLOR, edgecolor='white', linewidth=0)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Land Cover Class', fontsize=11, fontweight='bold')
    ax.set_ylabel('AUC-ROC',          fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.15); ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    plt.tight_layout()
    save('ROC_31_AUC_PerClass_RF_vs_GTB.png')

    # ── PR_32 — RF Precision-Recall per Class ────────────────
    fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
    for i in range(N):
        prec, rec, _ = precision_recall_curve(y_bin_rf[:, i], probs_rf[:, i])
        ap = average_precision_score(y_bin_rf[:, i], probs_rf[:, i])
        ax.plot(rec, prec, color=CLASS_COLORS[i], linewidth=2,
                label=f'{CLASS_NAMES[i]}  AP={ap:.3f}')
    ax.set_xlabel('Recall',    fontsize=11, fontweight='bold')
    ax.set_ylabel('Precision', fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.legend(fontsize=9, loc='lower left')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    plt.tight_layout()
    save('PR_32_RF_PrecisionRecallCurve_PerClass.png')

    # ── PR_33 — GTB Precision-Recall per Class ───────────────
    fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
    for i in range(N):
        prec, rec, _ = precision_recall_curve(y_bin_gtb[:, i], probs_gtb[:, i])
        ap = average_precision_score(y_bin_gtb[:, i], probs_gtb[:, i])
        ax.plot(rec, prec, color=CLASS_COLORS[i], linewidth=2,
                label=f'{CLASS_NAMES[i]}  AP={ap:.3f}')
    ax.set_xlabel('Recall',    fontsize=11, fontweight='bold')
    ax.set_ylabel('Precision', fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.legend(fontsize=9, loc='lower left')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    plt.tight_layout()
    save('PR_33_GTB_PrecisionRecallCurve_PerClass.png')

    # ── PR_34 — Average Precision Bar (RF vs GTB) ────────────
    ap_rf  = [average_precision_score(y_bin_rf[:,  i], probs_rf[:,  i]) for i in range(N)]
    ap_gtb = [average_precision_score(y_bin_gtb[:, i], probs_gtb[:, i]) for i in range(N)]
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
    b1 = ax.bar(x - w/2, ap_rf,  w, label='Random Forest',
                color=RF_COLOR,  edgecolor='white', linewidth=0)
    b2 = ax.bar(x + w/2, ap_gtb, w, label='Gradient Tree Boost',
                color=GTB_COLOR, edgecolor='white', linewidth=0)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, fontsize=10)
    ax.set_xlabel('Land Cover Class',       fontsize=11, fontweight='bold')
    ax.set_ylabel('Average Precision (AP)', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.15); ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    plt.tight_layout()
    save('PR_34_AveragePrecision_PerClass_RF_vs_GTB.png')

    print("  ✅ ROC and PR curves complete.")

except Exception as e:
    print(f"\n  ⚠️  ROC/PR curves skipped — {e}")
    print("     Ensure Cell 13 has run and GEE MULTIPROBABILITY is supported for both classifiers.")

# ─────────────────────────────────────────────
print(f"\n✅ All plots saved to  {OUT}/")
print(f"   Total files: {len(os.listdir(OUT))}")

Fetching metrics from GEE...
✅ All GEE metrics fetched.

  ✅  CM_01_RF_ConfusionMatrix_Counts.png
  ✅  CM_02_GTB_ConfusionMatrix_Counts.png
  ✅  CM_03_RF_ConfusionMatrix_Normalized.png
  ✅  CM_04_GTB_ConfusionMatrix_Normalized.png
  ✅  ACC_05_OverallAccuracy_RF_vs_GTB.png
  ✅  ACC_06_Kappa_RF_vs_GTB.png
  ✅  PA_07_RF_ProducerConsumerAccuracy_PerClass.png
  ✅  PA_08_GTB_ProducerConsumerAccuracy_PerClass.png
  ✅  PA_09_ProducersAccuracy_RF_vs_GTB.png
  ✅  PA_10_ConsumersAccuracy_RF_vs_GTB.png
  ✅  F1_11_RF_F1Score_PerClass.png
  ✅  F1_12_GTB_F1Score_PerClass.png
  ✅  F1_13_F1Score_RF_vs_GTB.png
  ✅  ERR_14_OmissionError_RF_vs_GTB.png
  ✅  ERR_15_CommissionError_RF_vs_GTB.png
  ✅  RAD_16_RF_RadarChart_AccuracyMetrics.png
  ✅  RAD_17_GTB_RadarChart_AccuracyMetrics.png
  ✅  HM_18_RF_MetricsHeatmap_PerClass.png
  ✅  HM_19_GTB_MetricsHeatmap_PerClass.png
  ✅  PRED_20_RF_TrueVsPredicted_ClassCounts.png
  ✅  PRED_21_GTB_TrueVsPredicted_ClassCounts.png
  ✅  PRED_22_PerClassPredictionAccuracy_RF_

In [21]:
# ============================================================
# DOWNLOAD ALL EVAL PLOTS AS ZIP
# ============================================================

import os
import zipfile
from google.colab import files

ZIP_PATH  = '/content/eval_plots_LULC_EastSikkim.zip'
PLOTS_DIR = '/content/eval_plots'

# Zip all plots
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(PLOTS_DIR)):
        if fname.endswith('.png'):
            zf.write(os.path.join(PLOTS_DIR, fname), fname)

total = len(zipfile.ZipFile(ZIP_PATH).namelist())
size  = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"✅ Zipped {total} plots  —  {size:.1f} MB")
print("⬇️  Starting download...")

files.download(ZIP_PATH)

✅ Zipped 34 plots  —  4.2 MB
⬇️  Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>